## Exercise 1

As a good computer science engineer, you know that your first step should be to load and understand the data you are about to work on. 

- Load the file containing the mails (use pandas)
- Show the following statistics about this file:
    - total number of mails
    - total number of mails corresponding to spams
    - total number of mails not corresponding to spams (hams)
    - the mean length of the mails
- Sort the dataset according to the row's indexes (see https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.sort_index.html)

In [2]:
# dataset: https://www.kaggle.com/datasets/venky73/spam-mails-dataset?resource=download

import pandas as pd

data = pd.read_csv(
    "TD6-files/spams.csv",
    sep=",",
    header="infer",
    index_col=0,
    encoding="utf-8"
)

data = data.sort_index()
data

,label,text,label_num
0,ham,Subject: christmas tree farm pictures\r\n,0
1,ham,"Subject: vastar resources , inc .\r\ngary , pr...",0
2,ham,Subject: calpine daily gas nomination\r\n- cal...,0
3,ham,Subject: re : issue\r\nfyi - see note below - ...,0
4,ham,Subject: meter 7268 nov allocation\r\nfyi .\r\...,0
...,...,...,...
5166,spam,Subject: our pro - forma invoice attached\r\nd...,1
5167,spam,Subject: str _ rndlen ( 2 - 4 ) } { extra _ ti...,1
5168,spam,Subject: check me out !\r\n61 bb\r\nhey derm\r...,1
5169,spam,Subject: hot jobs\r\nglobal marketing specialt...,1


In [18]:
def count_mails(label_list):
    """Count the number of mails, hams and spams in dataframe"""
    hams = 0
    spams = 0
    
    for i,label in enumerate(label_list):
        if label == "ham" or label == 0:
            hams += 1
        elif label == "spam" or label == 1:
            spams += 1
        else:
            print(f"Bad label for mail in row {i}: {label}")
    
    print(f"Number of mails:\t {len(label_list)}")
    print(f"Number of hams: \t {hams}")
    print(f"Number of spams:\t {spams}")


count_mails(data["label"])

mails_len = [len(i) for i in data["text"]]
mean_len = round(sum(mails_len)/len(mails_len),2)

print(f"Mean mail length:\t {mean_len}")

Number of mails:	 3750
Number of hams: 	 2518
Number of spams:	 1232
Mean mail length:	 912.27


## Exercise 2

You wonder if your predecessor did the basics steps when creating the dataset you are working on. For instance, did he remove every duplicates in this dataset ?

- Search for mails with the exact same text (we call this kind of search an exact match search)
    - Ouch, you found around 178 duplicates ! Clearly, this dataset was not well-handled by your predecessor
- To investigate further, you chose to perform a cosine similarity test between every mails to find every duplicates:
    - import the required functions from scikit-learn
    - perform a similarity check between each possible pair of mail (we call this a fuzzy search)
    - do not hesitate to check the notebooks we explored during the lecture on packages 
- Remove every mail with a cosine similarity of 0.9 or more with another mail
    - At least ! You removed around 1,421 mails from this dataset, it will be easier to process now
- Make sure the dataset is still usable - print the total number of mails, spams and hams

In [8]:
seen = []
duplicates = []

for i,mail in enumerate(data["text"]):
    if mail not in seen:
        seen.append(mail)
    else:
        duplicates.append(i)

print(len(duplicates))

0


In [9]:
from sklearn.metrics import pairwise_kernels
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer()
X = vectorizer.fit_transform(data["text"])

sim_matrice = pairwise_kernels(X,metric="cosine")

print(sim_matrice)

[[1.         0.03717915 0.06157491 ... 0.16903085 0.09534626 0.09534626]
 [0.03717915 1.         0.50650827 ... 0.03927765 0.04874228 0.04431116]
 [0.06157491 0.50650827 1.         ... 0.05204029 0.04403203 0.0513707 ]
 ...
 [0.16903085 0.03927765 0.05204029 ... 1.         0.0805823  0.0805823 ]
 [0.09534626 0.04874228 0.04403203 ... 0.0805823  1.         0.09090909]
 [0.09534626 0.04431116 0.0513707  ... 0.0805823  0.09090909 1.        ]]


In [10]:
# Careful: when comparing mails with a cosine similarity matrice, you will compare a mail with itself in each row !

ids_to_remove = set()

for i,sim_row in enumerate(sim_matrice): # if j == i, do not process
    for j,score in enumerate(sim_row):

        if score > 0.9 and i != j:
            ids_to_remove.add(j)

print(len(ids_to_remove))
data = data.drop(list(ids_to_remove))

0


In [11]:
count_mails(data["label"])

Number of mails:	 3750
Number of hams: 	 2518
Number of spams:	 1232


## Exercise 3

Your data is now ready to be used to train a Machine Learning Model ! The goal for this model is to classify each mail in the "spam" or "ham" category. 

- Separate the text and the label of the mails in two lists
- Use scikit-learn's "train_test_split" function to create a train partition and a test partition
- Check for the number of mails, hams and spams in each partition
- A Machine Learning Model usually take vectors as inputs:
    - you MUST apply a vectorizer to your data
    - first step: import and initiate a vectorizer (see the last lecture's notebooks)
    - second step: apply it on each partition (on the texts of the mails) 
- Now is the time to train your first model: a linear regression might do the trick
    - import the model with scikit-learn
    - train it on the train partition and check the accuracy
    - test it on the test partition and check the accuracy
- You want to compare your model with more models
    - chose two models from the following list: https://scikit-learn.org/stable/supervised_learning.html
    - reproduce the experiment you made for the linear regression, but with these models instead
- Perform a cross-validation on your best model

In [14]:
from sklearn.model_selection import train_test_split

x = list(data["text"])
y = list(data["label_num"])

x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size=0.3, random_state=42
)

print("Train partition:")
count_mails(y_train)

print("\nTest partition:")
count_mails(y_test)

Train partition:
Number of mails:	 2625
Number of hams: 	 1737
Number of spams:	 888

Test partition:
Number of mails:	 1125
Number of hams: 	 781
Number of spams:	 344


In [15]:
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer()

x_train = vectorizer.fit_transform(x_train)
x_test = vectorizer.transform(x_test)

In [16]:
from sklearn.linear_model import LinearRegression

model = LinearRegression()    # we initialize the model
model.fit(x_train, y_train)   # we give training data to the model
model.score(x_train, y_train) # we check the accuracy of our model on the train partition 

0.9999916278103443

In [17]:
print(model.predict(x_test))
print(model.score(x_test,y_test))

[ 0.00763143  3.78953005  0.87166445 ...  0.26741568 -0.30723543
 -0.2089584 ]
-0.9336045309273164


In [20]:
from sklearn import svm

model = svm.SVC()
model.fit(x_train, y_train)   # we give training data to the model
model.score(x_train, y_train) # we check the accuracy of our model on the train partition 

0.9611428571428572

In [21]:
print(model.predict(x_test))
print(model.score(x_test,y_test))

[0 0 1 ... 0 0 0]
0.9413333333333334


In [22]:
from sklearn.linear_model import SGDClassifier

model = SGDClassifier(loss="hinge", penalty="l2", max_iter=5)
model.fit(x_train, y_train)   # we give training data to the model
model.score(x_train, y_train) # we check the accuracy of our model on the train partition 

/home/jabi/Documents/venv/idmc/lib/python3.14/site-packages/sklearn/linear_model/_stochastic_gradient.py:741: ConvergenceWarning: Maximum number of iteration reached before convergence. Consider increasing max_iter to improve the fit.
  warnings.warn(


0.992

In [23]:
print(model.predict(x_test))
print(model.score(x_test,y_test))

[0 0 1 ... 1 0 0]
0.9573333333333334


In [32]:
from sklearn.model_selection import cross_val_score

x_cross = vectorizer.transform(data["text"])
y_cross = data["label_num"]

scores = cross_val_score( # what do we mean by cross-validation ?
    model,
    x_cross,
    y_cross,
    cv=5                  # what does this number correspond to ?
)

print(scores)
print(scores.mean())

[0.94266667 0.95466667 0.95333333 0.96       0.93333333]
0.9488


/home/jabi/Documents/venv/idmc/lib/python3.14/site-packages/sklearn/linear_model/_stochastic_gradient.py:741: ConvergenceWarning: Maximum number of iteration reached before convergence. Consider increasing max_iter to improve the fit.
  warnings.warn(
/home/jabi/Documents/venv/idmc/lib/python3.14/site-packages/sklearn/linear_model/_stochastic_gradient.py:741: ConvergenceWarning: Maximum number of iteration reached before convergence. Consider increasing max_iter to improve the fit.
  warnings.warn(
/home/jabi/Documents/venv/idmc/lib/python3.14/site-packages/sklearn/linear_model/_stochastic_gradient.py:741: ConvergenceWarning: Maximum number of iteration reached before convergence. Consider increasing max_iter to improve the fit.
  warnings.warn(
/home/jabi/Documents/venv/idmc/lib/python3.14/site-packages/sklearn/linear_model/_stochastic_gradient.py:741: ConvergenceWarning: Maximum number of iteration reached before convergence. Consider increasing max_iter to improve the fit.
  warning